In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tensor-network error mitigation (TEM): Isang Qiskit Function ng Algorithmiq
> **Note:** Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status pa ito at maaaring magbago.


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Pangkalahatang-ideya
Ang Tensor-network Error Mitigation (TEM) method ng Algorithmiq ay isang hybrid
quantum-classical algorithm na dinisenyo para sa pagsasagawa ng noise mitigation nang buo sa
classical post-processing stage. Sa TEM, maaaring kalkulahin ng gumagamit ang
mga expected value ng mga observable habang inaaayos ang mga hindi maiiwasang error na dulot ng ingay
na nangyayari sa quantum hardware, na may mas mataas na katumpakan at kahusayan sa gastos —
kaya naman ito ay isang napaka-kaakit-akit na pagpipilian para sa mga quantum researcher at mga practitioner sa industriya.

Ang paraan ay binubuo ng pagtatayo ng isang tensor network na kumakatawan sa kabaligtaran ng
pandaigdigang noise channel na nakaka-apekto sa estado ng quantum processor, at pagkatapos ay
inilalapat ang mapa sa mga informationally complete na measurement outcome na nakuha mula sa
maingay na estado upang makakuha ng mga walang kinikilingang estimator para sa mga observable.

Bilang kalamangan, ginagamit ng TEM ang mga informationally complete na sukat upang magbigay ng
access sa isang malawak na hanay ng mga mitigated expected value ng mga observable at mayroon itong
pinakamainam na sampling overhead sa quantum hardware, ayon sa paglalarawan ni Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), at Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Ang
measurement overhead ay tumutukoy sa bilang ng karagdagang sukat na kailangan para
magsagawa ng mahusay na error mitigation, isang kritikal na salik sa pagiging posible ng
mga quantum computation. Kaya naman, ang TEM ay may potensyal na paganahin ang quantum
advantage sa mga kumplikadong sitwasyon, tulad ng mga aplikasyon sa mga larangan ng quantum
chaos, many-body physics, Hubbard dynamics, at mga simulation ng small molecule chemistry.

Ang mga pangunahing tampok at benepisyo ng TEM ay maaaring ibuod bilang:

1.  **Pinakamainam na measurement overhead**: Ang TEM ay optimal kaugnay ng
[mga theoretical bound](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
ibig sabihin, walang paraan ang makakamit ng mas maliit na measurement overhead. Sa madaling
salita, ang TEM ay nangangailangan ng pinakamaliit na bilang ng karagdagang sukat para sa
error mitigation. Nangangahulugan din ito na gumagamit ang TEM ng pinakamaliit na quantum runtime.
2.  **Kahusayan sa gastos**: Dahil ang TEM ay humahawak ng noise mitigation nang buo sa
post-processing stage, walang pangangailangan na magdagdag ng mga karagdagang Circuit sa quantum
computer, na hindi lamang nagpapamura ng kalkulasyon kundi nagbabawas din ng
panganib ng pagdaragdag ng mga karagdagang error dahil sa mga pagkukulang ng quantum
devices.
3.  **Pagtatantya ng maraming observable**: Salamat sa mga informationally-complete
na sukat, mahusay na tinatantya ng TEM ang maraming observable gamit ang parehong
measurement data mula sa quantum computer.
4.  **Measurement error mitigation**: Kasama rin sa TEM Qiskit Function ang isang
[proprietary measurement error mitigation method](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
na kayang bawasan nang malaki ang mga readout error pagkatapos ng maikling calibration run.
5.  **Katumpakan**: Ang TEM ay malaki ang naiaambag sa katumpakan at pagiging maaasahan ng
mga digital quantum simulation, na ginagawang mas tumpak at mapagkakatiwalaan ang mga quantum algorithm.
## Paglalarawan
Ang TEM function ay nagbibigay-daan sa iyo na makakuha ng mga error-mitigated expected value para sa
maraming observable sa isang quantum Circuit na may minimal na sampling overhead. Ang
Circuit ay sinusukat gamit ang isang informationally complete positive operator-valued
measure (IC-POVM), at ang mga nakolektang measurement outcome ay pinoproseso sa isang
classical computer. Ang sukat na ito ay ginagamit para magsagawa ng mga tensor network
na paraan at bumuo ng noise-inversion map. Inilalapat ng function ang isang mapa na ganap na
binabaligtad ang buong maingay na Circuit gamit ang mga tensor network upang kumatawan sa mga maingay
na layer.

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Error-mitigated estimation of an observable O via post-processing measurement outcomes of the noisy quantum processor. U and N denote an ideal quantum operation and the associated noise map, which can be generally non-local (and extended to grey boxes). D stands for a tensor of operators that are dual to the effects in the IC measurement. The noise mitigation module M is a tensor network that is efficiently contracted from the middle out. The first iteration of the contraction is represented by the dotted purple line, the second one by the dashed line, and the third one by the solid line.")

Kapag naisumite na ang mga Circuit sa function, sila ay ita-transpile at
io-optimize upang mabawasan ang bilang ng mga layer na may two-qubit gates (ang mga Gate
na mas maingay sa mga quantum device). Ang ingay na nakaka-apekto sa mga layer ay natututo sa pamamagitan ng
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
gamit ang isang sparse Pauli-Lindblad noise model ayon sa paglalarawan ni E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Ang noise model ay isang tumpak na paglalarawan ng ingay sa device na kayang
makuha ang mga banayad na katangian, kabilang ang qubit cross-talk. Gayunpaman, ang ingay sa mga
device ay maaaring mag-fluctuate at mag-drift at ang natutunan na ingay ay maaaring hindi tumpak sa
oras na isinasagawa ang pagtatantya. Maaari itong magresulta sa mga hindi tumpak na resulta.
## Pagsisimula
Mag-authenticate gamit ang iyong [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/), at piliin ang TEM function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na [na-save mo na ang iyong account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na kapaligiran.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Halimbawa
Ipinapakita ng sumusunod na snippet ang isang halimbawa kung saan ginagamit ang TEM upang kalkulahin ang mga expected value ng isang observable sa isang simpleng quantum Circuit.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Gamitin ang mga Qiskit Serverless API upang suriin ang status ng iyong Qiskit Function workload:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Maaari mong ibalik ang mga resulta tulad nito:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Ang inaasahang halaga para sa noiseless Circuit para sa ibinigay na operator ay dapat na mga `0.18409094298943401`.
## Mga Input
**Mga Parameter**

Pangalan | Uri | Paglalarawan | Kinakailangan | Default | Halimbawa
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Isang iterable ng mga PUB-like (primitive unified bloc) na object, tulad ng mga tuple na `(circuit, observables)` o `(circuit, observables, parameters, precision)`. Tingnan ang [Pangkalahatang-ideya ng mga PUB](/guides/primitive-input-output#overview-of-pubs) para sa karagdagang impormasyon. Kung ipinasa ang isang non-ISA Circuit, ito ay ita-transpile gamit ang mga pinakamainam na setting. Kung ipinasa ang isang ISA Circuit, hindi ito ita-transpile; sa kasong ito, ang observable ay dapat na tinukoy sa buong QPU. | Oo | N/A | (circuit, observables)
`backend_name` | str | Pangalan ng Backend na gagamitin para sa query.| Hindi | Kung hindi ibinigay, gagamitin ang pinaka-hindi abala na Backend. | "ibm_fez"
`options` | dict | Mga input na opsyon. Tingnan ang seksyong `Options` para sa karagdagang detalye. | Hindi | Tingnan ang seksyong `Options` para sa karagdagang detalye.| {"max_bond_dimension": 100\}

> **Caution:** Ang TEM ay kasalukuyang may mga sumusunod na limitasyon:
> 
>   - Hindi sinusuportahan ang mga parametrized Circuit. Ang arguments na parameter ay dapat itakda sa `None` kung tinukoy ang precision. Aalisin ang limitasyong ito sa mga susunod na bersyon.
>   - Sinusuportahan lamang ang mga Circuit na walang loops. Aalisin ang limitasyong ito sa mga susunod na bersyon.
>   - Hindi sinusuportahan ang mga non-unitary Gate, tulad ng reset, measure, at lahat ng uri ng control flow. Idadagdag ang suporta para sa reset sa mga paparating na release.
### Mga Opsyon
Isang dictionary na naglalaman ng mga advanced na opsyon para sa TEM. Ang dictionary ay maaaring maglaman ng mga key sa sumusunod na talahanayan. Kung hindi ibinibigay ang alinman sa mga opsyon, gagamitin ang default na halaga na nakalista sa talahanayan. Ang mga default na halaga ay angkop para sa karaniwang paggamit ng TEM.

Pangalan | Mga Pagpipilian | Paglalarawan | Default
-- | -- | -- | --
`tem_max_bond_dimension` | int | Ang maximum bond dimension na gagamitin para sa mga tensor network. | 500 |
`tem_compression_cutoff` | float | Ang cutoff na halaga na gagamitin para sa mga tensor network. | 1e-16
`compute_shadows_bias_from_observable` | bool | Isang boolean flag na nagpapahiwatig kung ang bias para sa classical shadows measurement protocol ay dapat i-tailor sa PUB observable o hindi. Kung False, gagamitin ang classical shadows protocol (pantay na posibilidad ng pagsukat ng Z, X, Y).| False
`shadows_bias` | np.ndarray | Ang bias na gagamitin para sa randomized classical shadows measurement protocol, isang 1d o 2d array na may laki na 3 o hugis (num_qubits, 3) ayon sa pagkakabanggit. Ang order ay ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int o `None` | Ang maximum na oras ng pagpapatakbo sa QPU sa mga segundo. Kung lalampas ang runtime sa halagang ito, makakansela ang job. Kung `None`, ilalapat ang default na limitasyon na itinakda ng Qiskit Runtime. | `None`
`num_randomizations` | int | Ang bilang ng mga randomization na gagamitin para sa noise learning at gate twirling. | 32
`max_layers_to_learn` | int | Ang maximum na bilang ng mga natatanging layer na matututunan. | 4
`mitigate_readout_error` | bool | Isang Boolean flag na nagpapahiwatig kung isasagawa ang readout error mitigation o hindi. | True
`num_readout_calibration_shots` | int | Ang bilang ng mga shot na gagamitin para sa readout error mitigation. | 10000
`default_precision` | float | Ang default na precision na gagamitin para sa mga PUB na hindi tinukoy ang precision. |0.02
`seed` | int o `None` | Itakda ang seed ng random number generator para sa reproducibility. Kung `None`, huwag itakda ang seed. | None
## Mga Output
Isang Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) na naglalaman ng TEM-mitigated na resulta. Ang resulta para sa bawat PUB ay ibinalik bilang isang [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) na naglalaman ng mga sumusunod na field:

Pangalan | Uri | Paglalarawan
-- | -- | --
data | DataBin | Isang Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) na naglalaman ng TEM mitigated observable at ng standard error nito. Ang DataBin ay may mga sumusunod na field: <ul><li>`evs`: Ang TEM-mitigated na halaga ng observable.</li><li>`stds`: Ang standard error ng TEM-mitigated na observable.</li></ul>
metadata | dict | Isang dictionary na naglalaman ng mga karagdagang resulta. Ang dictionary ay naglalaman ng mga sumusunod na key: <ul><li>`"evs_non_mitigated"`: Ang halaga ng observable nang walang error mitigation.</li><li>`"stds_non_mitigated"`: Ang standard error ng resulta nang walang error mitigation.</li><li>`"evs_mitigated_no_readout_mitigation"`: Ang halaga ng observable na may error mitigation ngunit walang readout error mitigation.</li><li>`"stds_mitigated_no_readout_mitigation"`: Ang standard error ng resulta na may error mitigation ngunit walang readout error mitigation.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: Ang halaga ng observable nang walang error mitigation ngunit may readout error mitigation.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: Ang standard error ng resulta nang walang error mitigation ngunit may readout error mitigation.</li></ul>
## Pagkuha ng mga mensahe ng error
Kung ang status ng iyong workload ay ERROR, gamitin ang job.result() para makuha ang mensahe ng error tulad ng sumusunod: